# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
* https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset loaded.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and fields by their @id
record_sets = dataset.record_sets

print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']} | Name: {rs.get('name', '<no name>')}")
    fields = rs.get('field', [])
    if fields:
        print("    Fields:")
        for f in fields:
            print(f"      - Field @id: {f['@id']} | Name: {f.get('name', '<no name>')} | Column: {f.get('column', '<no column>')}")
    else:
        print("    No fields listed.")
    print()

# Preview first few records from each record set
for rs in record_sets:
    rsid = rs['@id']
    print(f"--- Preview records for RecordSet @id: {rsid} ---")
    try:
        for i, rec in enumerate(dataset.records(record_set=rsid)):
            print(rec)
            if i >= 2: break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print()

## 3. Data Extraction

Load data from each record set into Pandas DataFrames for analysis.
Each record set and field is referenced by its `@id`.

In [ ]:
# Store all dataframes, referenced by record set @id
dataframes = {}

# Build a list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame for RecordSet @id: {rsid}")
        print("  Columns:", df.columns.tolist())
        print("  Preview:")
        print(df.head())
    except Exception as e:
        print(f"  Could not create DataFrame for @id: {rsid} ({e})")
    print()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping by attributes. All fields are referenced using their `@id`.

In [ ]:
# Choose a record set for EDA
# Example: Table 1 contains clinical features like age, height, and presence of symptoms
table1_record_set_id = None
for rs in record_sets:
    if 'Table 1' in rs.get('name', '') or 'Clinical Features' in rs.get('name', ''):
        table1_record_set_id = rs['@id']
        break
if not table1_record_set_id:
    table1_record_set_id = record_set_ids[0] # fallback: use the first

df = dataframes[table1_record_set_id]
print(f"Working with DataFrame for RecordSet @id: {table1_record_set_id}")

# Identify numeric fields (by columns with float/int dtype)
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_cols:
    numeric_field_id = numeric_cols[0] # Example: first numeric column
    print(f"Numeric field chosen for analysis: {numeric_field_id}")
else:
    print("No numeric fields found; EDA on categorical columns.")

# Example threshold filtering if numeric_field_id exists
if numeric_cols:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} (@id) > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping - try a categorical column
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (@id):")
        print(grouped_df.head())

## 5. Visualization

Visualize distributions or relationships between fields in the dataset. All axes/fields reference the column (field) `@id`.

In [ ]:
# Visualization: Histogram of numeric field (referenced by @id)
if numeric_cols:
    plt.figure(figsize=(6,4))
    df[numeric_field_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id} (Field @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Visualization: Boxplot by group field
if numeric_cols and group_field_id:
    plt.figure(figsize=(6,4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (Field @id)")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we explored the FAIR^2 dataset on genotype–phenotype heterogeneity and infertility in NC-21OHD using `mlcroissant`. We demonstrated how to load structured tabulations via Croissant, reviewed available record sets and fields by their `@id`, and extracted, analyzed, and visualized sample clinical and genetic features. Further research can build on this template to expand analyses or cross-reference other rare disease datasets using the Croissant standard.